# OceanWatch Analytics · Entrega 2
Consola, gobierno de la capa oro, detección de anomalías de velocidad y publicación por Delta Sharing. Este notebook continúa la tarea 04 del equipo 8. El registro se realiza en el workspace institucional. El consumidor del bono tiene su propio notebook para Free Edition.

En `modo=pipeline` se calculan las anomalías de las fechas nuevas o corregidas. Los otros modos se ejecutan manualmente una vez o para recoger evidencia. La guía indica el orden y los permisos de cada operación.

In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))

In [ ]:
%run ./_config

In [ ]:
dbutils.widgets.dropdown("modo", "pipeline", ["pipeline","gobernanza","tablero","configurar_job","verificar","compartir"])
dbutils.widgets.text("grupo_lectura", "", "Grupo de consumidores existente")
dbutils.widgets.text("warehouse_id", "", "ID del SQL warehouse")
dbutils.widgets.text("ruta_dashboard", "", "Ruta absoluta a OceanWatch_Institucional.lvdash.json")
dbutils.widgets.text("ruta_src", "", "Carpeta de notebooks del workspace (opcional)")
modo = dbutils.widgets.get("modo")
print("Operación", modo, "· día", fecha)

## Detección de anomalías de velocidad
La unidad de evaluación es un buque en un día UTC. La SOG media procede del AIS. La segunda velocidad divide la distancia acumulada entre posiciones por el tiempo transcurrido entre primera y última señal, incluyendo pausas. Su diferencia ayuda a detectar saltos de posición o velocidades inusuales aun cuando la SOG de cada mensaje esté dentro del rango permitido. Un aviso requiere revisión.

Se usan las dos velocidades y la brecha absoluta, con transformación logarítmica para la velocidad por tramos y la brecha. Isolation Forest aprende con los 14 días anteriores disponibles. El corte es estrictamente anterior al día puntuado. La muestra de entrenamiento tiene un límite de 30.000 trayectorias, un mínimo de 100 y semilla 42. El contraste es el percentil 99 de SOG del entrenamiento. Se registran cobertura, avisos y coincidencias. La fuente carece de etiquetas de anomalías confirmadas.

La versión se registra con firma y ejemplo de entrada. En inferencia se vuelve a cargar por versión. El alias `champion` identifica la versión vigente. Los días con menos de diez minutos entre señales quedan como `traza_insuficiente`; un arranque sin historia queda como `sin_historial`.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import IsolationForest

VARIABLES = ["sog_media", "log_velocidad_tramos", "log_brecha"]
PARAM = dict(n_estimators=120, contamination=0.015, max_samples="auto", random_state=42, n_jobs=1)
MIN_ENTRENO, MAX_ENTRENO = 100, 30000
VERSION_FEATURES = "velocidad-v1"


def matriz(df):
    """La escala logarítmica conserva extremos cinemáticos sin desbordar las distancias."""
    return pd.DataFrame({"sog_media": df.sog_media.astype("float64"),
                         "log_velocidad_tramos": np.log1p(df.velocidad_tramos.astype("float64")),
                         "log_brecha": np.log1p(df.brecha.astype("float64"))})[VARIABLES]


def huella_muestra(pdf):
    import json
    orden = pdf.sort_values(["fecha", "MMSI"])[["fecha", "MMSI", "sog_media", "velocidad_tramos", "brecha"]]
    # El redondeo también se aplica a los atributos del modelo, para que una
    # distinta asociación de sumas en Spark no cree versiones por ruido numérico.
    binario = orden.to_csv(index=False, float_format="%.6f", lineterminator="\n").encode()
    receta = json.dumps([VERSION_FEATURES, PARAM, sklearn.__version__], sort_keys=True).encode()
    return hashlib.sha256(receta + binario).hexdigest()


def entrenar(pdf):
    modelo = IsolationForest(**PARAM).fit(matriz(pdf))
    return modelo, float(pdf["sog_media"].quantile(0.99))


def puntuaciones(modelo, pdf):
    return -modelo.decision_function(matriz(pdf))

In [ ]:
def atributos(sesion, tabla):
    from pyspark.sql import functions as F
    t = sesion.table(tabla)
    horas = (F.col("fin").cast("long") - F.col("inicio").cast("long")) / 3600.0
    util = (F.col("n_posiciones") >= 2) & (horas >= 1 / 6) & F.col("sog_media").isNotNull()
    return (t.withColumn("velocidad_tramos", F.when(util, F.round(F.col("distancia_km") / (1.852 * horas), 6)))
            .withColumn("sog_media", F.round("sog_media", 6))
            .withColumn("brecha", F.round(F.abs(F.col("velocidad_tramos") - F.col("sog_media")), 6))
            .select("fecha", "MMSI", "n_posiciones", "sog_media", "velocidad_tramos", "brecha"))


def version_para_muestra(cliente, experimento, nombre, muestra, dia):
    import mlflow
    from mlflow.models import infer_signature
    from mlflow.exceptions import MlflowException
    firma = huella_muestra(muestra)
    try:
        versiones = cliente.search_model_versions(f"name = '{nombre}'", max_results=1000)
    except MlflowException as exc:
        if exc.error_code not in {"RESOURCE_DOES_NOT_EXIST", "NOT_FOUND"}:
            raise
        versiones = []
    for mv in versiones:
        if mv.run_id:
            prev = cliente.get_run(mv.run_id)
            if prev.data.tags.get("huella") == firma:
                return str(mv.version), mv.run_id, float(prev.data.params["umbral_base_p99"])
    modelo, q99 = entrenar(muestra)
    ej = matriz(muestra.iloc[:5])
    with mlflow.start_run(experiment_id=experimento.experiment_id, run_name="Entrenamiento " + str(dia)) as run:
        mlflow.set_tags({"huella": firma, "version_atributos": VERSION_FEATURES, "fuente": GOLD_TRAY,
                         "unidad": "buque-dia", "fecha_corte": str(dia)})
        mlflow.log_params({**PARAM, "inicio_train": str(min(muestra.fecha)), "fin_train": str(max(muestra.fecha)),
                           "umbral_base_p99": q99, "variables": ",".join(VARIABLES),
                           "sklearn": sklearn.__version__, "ventana_dias": 14})
        mlflow.log_metric("filas_entrenamiento", len(muestra))
        dataset = mlflow.data.from_pandas(muestra[["fecha","MMSI","sog_media","velocidad_tramos","brecha"]],
                                          name="trayectorias_oro", digest=firma[:16])
        mlflow.log_input(dataset, context="training")
        mlflow.log_dict({"variables": VARIABLES, "llave": ["fecha", "MMSI"], "umbral_alerta": 0.0,
                         "huella": firma}, "contrato_entrada.json")
        datos = {"name": "detector"} if int(mlflow.__version__.split(".")[0]) >= 3 else {"artifact_path": "detector"}
        info = mlflow.sklearn.log_model(modelo, signature=infer_signature(ej, modelo.predict(ej)), input_example=ej, **datos)
        mv = mlflow.register_model(info.model_uri, nombre)
        for principal in ("`mine-group8`", "`h.benitez@uniandes.edu.co`", "`l.galvan@uniandes.edu.co`"):
            spark.sql(f"GRANT EXECUTE ON MODEL {nombre} TO {principal}")
        return str(mv.version), run.info.run_id, q99


def puntuar_dia(fecha_dia, registro, ruta_experimento):
    fecha = fecha_dia.isoformat()
    import datetime as dt
    import mlflow
    from mlflow import MlflowClient
    from mlflow.exceptions import MlflowException
    from pyspark.sql import functions as F

    inicio = fecha_dia - dt.timedelta(days=14)
    base = atributos(spark, GOLD_TRAY)
    jornada = base.where(F.col("fecha") == F.lit(fecha_dia))
    if jornada.limit(1).count() == 0:
        raise ValueError(f"Falta la trayectoria oro de {fecha}. Ejecuta 04_gold para la fecha.")
    entreno = (base.where((F.col("fecha") >= F.lit(inicio)) & (F.col("fecha") < F.lit(fecha_dia))
                          & F.col("brecha").isNotNull())
               .orderBy(F.xxhash64("fecha", "MMSI"), "fecha", "MMSI")
               .limit(MAX_ENTRENO).toPandas())
    columnas = ["fecha","MMSI","n_posiciones","sog_media","velocidad_tramos","brecha",
                "score","es_anomalia","estado","modelo_version","run_id"]
    if len(entreno) < MIN_ENTRENO:
        salida = (jornada.withColumn("score", F.lit(None).cast("double"))
                  .withColumn("es_anomalia", F.lit(None).cast("boolean"))
                  .withColumn("estado", F.when(F.col("brecha").isNull(), "traza_insuficiente").otherwise("sin_historial"))
                  .withColumn("modelo_version", F.lit(None).cast("string"))
                  .withColumn("run_id", F.lit(None).cast("string")))
        print(f"Arranque histórico de {fecha}: {len(entreno)} ejemplos anteriores; se requieren {MIN_ENTRENO}.")
    else:
        mlflow.set_registry_uri(registro)
        exp = mlflow.set_experiment(ruta_experimento or "/Shared/OceanWatch_eq8_velocidad")
        cliente = MlflowClient()
        version, run_id, q99 = version_para_muestra(cliente, exp, MODELO, entreno, fecha_dia)
        # Un backfill usa su versión por fecha sin desplazar una versión más reciente.
        otro_corte = ""
        if "champion" in (cliente.get_registered_model(MODELO).aliases or {}):
            anterior = cliente.get_model_version_by_alias(MODELO, "champion")
            otro_corte = cliente.get_run(anterior.run_id).data.tags.get("fecha_corte", "")
        if fecha >= otro_corte:
            cliente.set_registered_model_alias(MODELO, "champion", version)
        modelo = mlflow.sklearn.load_model(f"models:/{MODELO}/{version}")
        @F.pandas_udf("double")
        def puntuar(sog: pd.Series, velocidad: pd.Series, brechas: pd.Series) -> pd.Series:
            bloque = pd.DataFrame({"sog_media":sog,"velocidad_tramos":velocidad,"brecha":brechas})
            elegibles = bloque.notna().all(axis=1)
            scores = pd.Series(np.nan, index=bloque.index, dtype="float64")
            if elegibles.any():
                scores.loc[elegibles] = puntuaciones(modelo, bloque.loc[elegibles])
            return scores
        salida = (jornada.withColumn("score", puntuar("sog_media", "velocidad_tramos", "brecha"))
                  .withColumn("es_anomalia", F.when(F.col("brecha").isNotNull(), F.col("score") > 0))
                  .withColumn("estado", F.when(F.col("brecha").isNull(), "traza_insuficiente").otherwise("evaluado"))
                  .withColumn("modelo_version", F.lit(version)).withColumn("run_id", F.lit(run_id)))
    (salida.select(*columnas).write.format("delta").mode("overwrite")
     .option("replaceWhere", f"fecha = '{fecha}'").saveAsTable(GOLD_ANOM))
    if len(entreno) >= MIN_ENTRENO:
        reales = spark.table(GOLD_ANOM).where(F.col("fecha") == F.lit(fecha_dia))
        met = reales.agg(F.count("*").alias("total"),F.count("es_anomalia").alias("evaluados"),
            F.count(F.when(F.col("es_anomalia"),1)).alias("alertas_modelo"),
            F.count(F.when((F.col("estado")=="evaluado") & (F.col("sog_media")>q99),1)).alias("alertas_base"),
            F.count(F.when(F.col("es_anomalia") & (F.col("sog_media")>q99),1)).alias("coinciden")).first()
        with mlflow.start_run(experiment_id=exp.experiment_id, run_name="Evaluación temporal " + fecha):
            mlflow.set_tags({"fecha_evaluacion":fecha,"run_entrenamiento":run_id,"modelo_version":version})
            mlflow.log_metrics({**met.asDict(),"cobertura":met.evaluados/met.total,
                                "tasa_alertas":met.alertas_modelo/max(1,met.evaluados)})
        print("Modelo", MODELO, "versión", version, "run", run_id, met.asDict())


def ejecutar_ml(registro="databricks-uc", ruta_experimento=None):
    # Si entra un mensaje tardío, cambian tanto su día como las ventanas posteriores que lo usaron.
    import json
    receta = hashlib.sha256(json.dumps([VERSION_FEATURES, PARAM, sklearn.__version__, MIN_ENTRENO, MAX_ENTRENO],sort_keys=True).encode()).hexdigest()
    pendientes = spark.sql(f"""WITH firmas AS (
       SELECT t.fecha, sha2(concat('{receta}',concat_ws('|',sort_array(collect_list(
          concat(cast(h.fecha as string),'=',cast(h.batch_plata as string)))))),256) revision
       FROM {CONTROL_ORO} t JOIN {CONTROL_ORO} h
       ON h.fecha BETWEEN date_sub(t.fecha,14) AND t.fecha GROUP BY t.fecha)
       SELECT f.fecha,f.revision FROM firmas f LEFT JOIN {CONTROL_ML} m ON f.fecha=m.fecha
       WHERE m.fecha IS NULL OR f.revision<>m.revision_oro ORDER BY f.fecha""").collect()
    for p in pendientes:
        puntuar_dia(p.fecha, registro, ruta_experimento)
        spark.sql(f"""MERGE INTO {CONTROL_ML} d USING (SELECT DATE '{p.fecha.isoformat()}' fecha,
             '{p.revision}' revision_oro,current_timestamp() publicado_en) s ON d.fecha=s.fecha
             WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    if not spark.table(CONTROL_ML).where(f"fecha = DATE '{fecha}'").take(1):
        raise ValueError(f"El día {fecha} aún no tiene cierre de oro. Ejecuta 04_gold.")
    print(f"Fechas de ML actualizadas: {len(pendientes)}. Día del job: {fecha}.")


In [ ]:
if modo == "pipeline":
    ejecutar_ml()

## Gobierno y contrato
El contrato en `docs/contrato_oro.md` especifica tipos, llaves, cobertura, reglas de acceso y las consultas de conciliación. Indica en `grupo_lectura` el nombre de un grupo analítico real de la universidad. Este modo requiere propiedad o permisos de gestión. `verificar` consulta las evidencias del linaje y del job.

In [ ]:
OROS = [GOLD_TRAY, GOLD_PUERTO, GOLD_TRAFICO, GOLD_ANOM]

def gobierno(lector):
    # El grupo de ingeniería ya existe en el catálogo institucional.
    grupo = "`mine-group8`"
    if lector and lector.strip().lower() == "mine-group8":
        raise ValueError("Indica un grupo analítico distinto del grupo de ingeniería.")
    if lector:
        # La comilla invertida se escapa como identificador de Unity Catalog.
        rol = "`" + lector.replace("`", "``") + "`"
        spark.sql(f"GRANT USE CATALOG ON CATALOG `{CATALOG}` TO {rol}")
        spark.sql(f"GRANT USE SCHEMA ON SCHEMA `{CATALOG}`.`{SCHEMA}` TO {rol}")
        # Las cuatro tablas concretas son el contrato público de este esquema.
        for t in OROS:
            spark.sql(f"GRANT SELECT ON TABLE {t} TO {rol}")
    else:
        print("El widget grupo_lectura está vacío. Se inspeccionan los permisos existentes.")
    dic = {
        "fecha":"Día UTC del evento AIS.", "MMSI":"Identificador AIS del buque, nueve dígitos.",
        "n_posiciones":"Mensajes válidos dentro de la unidad de esta tabla.", "inicio":"Primera señal UTC.",
        "fin":"Última señal UTC.", "lat_inicio":"Latitud inicial, grados WGS84.", "lon_inicio":"Longitud inicial, grados WGS84.",
        "lat_fin":"Latitud final, grados WGS84.", "lon_fin":"Longitud final, grados WGS84.",
        "distancia_km":"Suma de tramos haversine diarios en kilómetros.", "sog_media":"Media de SOG por mensaje, en nudos.",
        "numero_puerto":"Número estable del World Port Index.", "puerto":"Nombre del puerto en WPI.", "pais":"País según WPI.",
        "hora":"Hora UTC entre 0 y 23.", "h3_cell":"Identificador hexadecimal H3 de resolución 8.",
        "n_buques":"MMSI distintos en la celda y hora.", "velocidad_tramos":"Distancia observada dividida por tiempo total, en nudos.",
        "brecha":"Diferencia absoluta de velocidades, en nudos.", "score":"Opuesto de decision_function; valor positivo indica alerta.",
        "es_anomalia":"Alerta del modelo. Nulo si el buque no fue evaluado.", "estado":"evaluado, sin_historial o traza_insuficiente.",
        "modelo_version":"Versión registrada utilizada para puntuar la jornada.", "run_id":"Run MLflow de entrenamiento."
    }
    for t in OROS:
        for columna in spark.table(t).columns:
            spark.sql(f"ALTER TABLE {t} ALTER COLUMN `{columna}` COMMENT '{dic[columna]}'")
        spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('oceanwatch.contrato'='2.0', "
                  "'oceanwatch.frecuencia'='diaria', 'oceanwatch.huso'='UTC', "
                  "'oceanwatch.responsable'='mine-group8')")
        display(spark.sql(f"SHOW GRANTS ON TABLE {t}"))
    display(spark.sql(f"SHOW GRANTS ON SCHEMA `{CATALOG}`.`{SCHEMA}`"))
    display(spark.sql(f"SHOW GRANTS ON TABLE {SILVER}"))


In [ ]:
if modo == "gobernanza":
    gobierno(dbutils.widgets.get("grupo_lectura").strip())

## Consola AI/BI
El archivo `dashboards/OceanWatch_Institucional.lvdash.json` define los conjuntos de datos y la interfaz. El selector de fecha actúa sobre el mapa de intensidad, el ranking, la presencia de buques y los casos de velocidad. Las visualizaciones consultan tablas de oro. Para publicar desde el notebook elige `tablero` e indica un SQL warehouse. También se puede importar el JSON desde Dashboards.

In [ ]:
def publicar_tablero(archivo, titulo, warehouse, copia=""):
    import json
    from pathlib import Path
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.dashboards import Dashboard

    w = WorkspaceClient()
    user = spark.sql("SELECT current_user()").first()[0]
    carpeta = f"/Users/{user}/OceanWatch"
    w.workspace.mkdirs(carpeta)
    caminos = [Path(copia)] if copia else []
    caminos += [Path.cwd().parent/"dashboards"/archivo, Path.cwd()/"dashboards"/archivo,
                Path.cwd()/archivo]
    ruta = next((p for p in caminos if p.is_file()), None)
    if ruta is None:
        raise FileNotFoundError(f"Indica en ruta_dashboard la ruta absoluta al archivo {archivo} del paquete.")
    definicion = json.loads(ruta.read_text(encoding="utf-8"))
    if not warehouse:
        candidatos = list(w.warehouses.list())
        if len(candidatos) != 1:
            raise ValueError("Selecciona un SQL warehouse accesible e indica su ID en warehouse_id.")
        warehouse = candidatos[0].id
    # Comprobar en el mismo warehouse que usará el lector del dashboard.
    from databricks.sdk.service import sql
    import datetime, time
    dia_widget = dbutils.widgets.get("fecha").strip()
    dia_sql = globals().get("fecha_dia", None) if dia_widget.upper()=="AUTO" else datetime.date.fromisoformat(dia_widget)
    if dia_sql is None:
        raise ValueError("Selecciona un día AIS explícito para publicar el tablero.")
    pruebas_sql = {}
    for ds in definicion["datasets"]:
        consulta = "SELECT * FROM (" + "".join(ds["queryLines"]) + ") ow LIMIT 1"
        st = w.statement_execution.execute_statement(consulta,warehouse,
            parameters=[sql.StatementParameterListItem(name="dia",type="DATE",value=dia_sql.isoformat())],
            wait_timeout="30s")
        limite = time.monotonic() + 300
        while st.status.state in {sql.StatementState.RUNNING,sql.StatementState.PENDING} and time.monotonic() < limite:
            time.sleep(3)
            st = w.statement_execution.get_statement(st.statement_id)
        if st.status.state != sql.StatementState.SUCCEEDED:
            detalle = st.status.error.message if st.status.error else st.status.state.value
            raise RuntimeError(f"Dataset {ds['name']}, sentencia {st.statement_id}: {detalle}")
        pruebas_sql[ds["name"]] = st.statement_id
    print("Consultas validadas en el warehouse:",pruebas_sql)
    existentes = [d for d in w.lakeview.list() if d.display_name == titulo and
                  ((d.parent_path or "").removeprefix("/Workspace") == carpeta or
                   str(d.path or "").removeprefix("/Workspace").rsplit("/",1)[0] == carpeta)]
    if len(existentes) > 1:
        raise ValueError(f"Hay varios dashboards con el nombre {titulo} en {carpeta}.")
    definicion = json.dumps(definicion, ensure_ascii=False)
    if existentes:
        previo = w.lakeview.get(existentes[0].dashboard_id)
        d = w.lakeview.update(previo.dashboard_id, Dashboard(display_name=titulo,
            etag=previo.etag, warehouse_id=warehouse, serialized_dashboard=definicion))
    else:
        d = w.lakeview.create(Dashboard(display_name=titulo, parent_path=carpeta, warehouse_id=warehouse,
                                         serialized_dashboard=definicion))
    w.lakeview.publish(d.dashboard_id, warehouse_id=warehouse, embed_credentials=False)
    w.lakeview.get_published(d.dashboard_id)
    print("Tablero publicado:", w.config.host.rstrip("/") + "/dashboardsv3/" + d.dashboard_id + "/published")
    return d.dashboard_id

In [ ]:
if modo == "tablero":
    publicar_tablero("OceanWatch_Institucional.lvdash.json", "OceanWatch · Operación marítima",
                     dbutils.widgets.get("warehouse_id").strip(), dbutils.widgets.get("ruta_dashboard").strip())

## Orquestación y verificación
El modo `configurar_job` actualiza el job existente, añade ML tras oro, mantiene el cómputo de las tareas y activa el calendario diario a las 06:00 de Bogotá. El modo `AUTO` reproduce un archivo histórico diferente por día desde 2023-06-08. Para repetir una fecha pasa `fecha=YYYY-MM-DD`. En la carpeta personal de Databricks se puede usar `ruta_src` para apuntar las cinco tareas a los notebooks importados.

In [ ]:
NOMBRE_JOB = "Orquestracion OceanWatch"

def job_equipo(w):
    candidatos = [j for j in w.jobs.list(name=NOMBRE_JOB) if j.settings.name == NOMBRE_JOB]
    if len(candidatos) != 1:
        raise ValueError(f"Esperaba exactamente un job visible llamado {NOMBRE_JOB}. Encontré {len(candidatos)}.")
    return w.jobs.get(candidatos[0].job_id)

def configurar_job(ruta_src):
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service import jobs
    from datetime import datetime
    from zoneinfo import ZoneInfo
    w = WorkspaceClient()
    j = job_equipo(w)
    previas = j.settings.tasks
    def buscar(stem):
        res = [t for t in previas if t.notebook_task and
               t.notebook_task.notebook_path.rsplit("/",1)[-1].removesuffix(".ipynb")==stem]
        if len(res)!=1:
            raise ValueError(f"Falta una tarea inequívoca para {stem} en el job del equipo.")
        return jobs.Task.from_dict(res[0].as_dict())
    nombres = ["01_ingesta","02_bronze","03_silver","04_gold"]
    cadena = [buscar(n) for n in nombres]
    clusters = [c for c in w.clusters.list() if c.cluster_name == "clase14-equipo8"]
    if len(clusters) != 1:
        raise ValueError("Se necesita acceso al cómputo clase14-equipo8 para configurar las cinco tareas.")
    prev5 = [t for t in previas if t.notebook_task and t.notebook_task.notebook_path.rsplit("/",1)[-1].removesuffix(".ipynb")=="05_oceanwatch"]
    if len(prev5)>1:
        raise ValueError("Hay más de una tarea 05 en el job.")
    if prev5:
        ultima=jobs.Task.from_dict(prev5[0].as_dict())
    else:
        ultima=jobs.Task.from_dict(cadena[-1].as_dict())
        ultima.task_key="oceanwatch_ml"
        if any(t.task_key==ultima.task_key for t in previas):
            raise ValueError("Ya existe otra tarea con la clave oceanwatch_ml.")
        extension=".ipynb" if ultima.notebook_task.notebook_path.endswith(".ipynb") else ""
        ultima.notebook_task.notebook_path=ultima.notebook_task.notebook_path.rsplit("/",1)[0]+"/05_oceanwatch"+extension
    cadena.append(ultima)
    for i,t in enumerate(cadena):
        if ruta_src:
            t.notebook_task.notebook_path=ruta_src.rstrip("/")+"/"+(nombres+["05_oceanwatch"])[i]
            from databricks.sdk.errors import NotFound
            try:
                w.workspace.get_status(t.notebook_task.notebook_path)
            except NotFound:
                t.notebook_task.notebook_path+=".ipynb"
                w.workspace.get_status(t.notebook_task.notebook_path)
            t.notebook_task.source=jobs.Source.WORKSPACE
        if i:
            deps = list(t.depends_on or [])
            # 05 clona el cómputo de 04. Su dependencia directa es 04.
            if i==4 and not prev5:
                deps=[]
            if cadena[i-1].task_key not in {d.task_key for d in deps}:
                deps.append(jobs.TaskDependency(task_key=cadena[i-1].task_key))
            t.depends_on=deps
        t.existing_cluster_id=clusters[0].cluster_id
        t.job_cluster_key=None
        t.new_cluster=None
        t.max_retries=2
        t.min_retry_interval_millis=60000
        t.retry_on_timeout=True
        p = dict(t.notebook_task.base_parameters or {})
        p.update(fecha="{{job.parameters.fecha}}",ancla_replay="{{job.parameters.ancla_replay}}",
                 inicio_run="{{job.start_time.iso_date}}")
        if i==4: p["modo"]="pipeline"
        t.notebook_task.base_parameters=p
    parametros={p.name:p.default for p in (j.settings.parameters or [])}
    parametros["fecha"]="AUTO"
    parametros.setdefault("ancla_replay",datetime.now(ZoneInfo("America/Bogota")).date().isoformat())
    settings = j.settings
    settings.tasks = cadena
    settings.parameters = [jobs.JobParameterDefinition(name=k, default=v) for k, v in parametros.items()]
    settings.schedule = jobs.CronSchedule(
        quartz_cron_expression="0 0 6 * * ?",
        timezone_id="America/Bogota",
        pause_status=jobs.PauseStatus.UNPAUSED,
    )
    settings.max_concurrent_runs = 1
    w.jobs.update(j.job_id, new_settings=settings)
    real=w.jobs.get(j.job_id)
    print("Job", real.job_id,real.settings.name,real.settings.schedule.as_dict())
    for t in real.settings.tasks:
        print(t.task_key,"←",[d.task_key for d in t.depends_on or []],"reintentos",t.max_retries)


In [ ]:
def verificar():
    t = f"DATE '{fecha}'"
    def valor(q): return spark.sql(q).first()[0]
    def claves(tabla, key):
        return valor(f"SELECT count(*) FROM (SELECT {key} FROM {tabla} WHERE fecha={t} GROUP BY {key} HAVING count(*)>1)")
    for tabla, llave in zip(OROS, ["MMSI","numero_puerto,MMSI","hora,h3_cell","MMSI"]):
        if claves(tabla,llave)!=0:
            raise ValueError(f"Llave repetida en {tabla}")
    plata=valor(f"SELECT count(*) FROM {SILVER} WHERE fecha_dia={t}")
    tray=valor(f"SELECT coalesce(sum(n_posiciones),0) FROM {GOLD_TRAY} WHERE fecha={t}")
    traf=valor(f"SELECT coalesce(sum(n_posiciones),0) FROM {GOLD_TRAFICO} WHERE fecha={t}")
    if not (plata>0 and plata==tray==traf):
        raise ValueError((plata,tray,traf))
    if valor(f"SELECT count(*) FROM {METRICAS_V2} WHERE entrada<>nuevas+duplicados_batch+duplicados_previos+cuarentena")!=0:
        raise ValueError("El balance de oceanwatch_metricas_silver_v2 no cierra")
    if valor(f"""SELECT count(*) FROM (SELECT batch FROM {METRICAS_V2}
                 GROUP BY batch HAVING count(*)>1)""")!=0:
        raise ValueError("Hay un batch repetido en las metricas de silver")
    faltan=valor(f"""SELECT count(*) FROM (SELECT MMSI FROM {GOLD_TRAY} WHERE fecha={t}
         EXCEPT SELECT MMSI FROM {GOLD_ANOM} WHERE fecha={t})""")
    if faltan!=0:
        raise ValueError(f"Faltan {faltan} evaluaciones")
    if valor(f"SELECT count(*) FROM {GOLD_TRAY} WHERE fecha={t}") != valor(f"SELECT count(*) FROM {GOLD_ANOM} WHERE fecha={t}"):
        raise ValueError("Trayectorias y anomalias no cubren los mismos buques")
    if valor(f"SELECT count(*) FROM (SELECT MMSI,BaseDateTime FROM {SILVER} WHERE fecha_dia={t} GROUP BY MMSI,BaseDateTime HAVING count(*)>1)")!=0:
        raise ValueError("Silver tiene mas de una fila por MMSI e instante")
    if valor(f"""SELECT count(*) FROM {GOLD_ANOM} WHERE fecha={t} AND
        (estado IS NULL OR estado NOT IN ('evaluado','traza_insuficiente','sin_historial')
         OR (estado='evaluado' AND (score IS NULL OR es_anomalia IS NULL OR modelo_version IS NULL OR run_id IS NULL))
         OR (estado<>'evaluado' AND (score IS NOT NULL OR es_anomalia IS NOT NULL)))""")!=0:
        raise ValueError("Hay evaluaciones con estado o score inconsistente")
    display(spark.sql(f"""SELECT estado,modelo_version,run_id,count(*) buques,
        count_if(es_anomalia) alertas FROM {GOLD_ANOM} WHERE fecha={t} GROUP BY ALL"""))
    print(fecha, "plata = posiciones en trayectorias = tráfico", plata)
    linaje, acceso_linaje = [], "disponible"
    try:
        consulta_linaje = spark.sql(f"""SELECT event_time,source_table_full_name,target_table_full_name
          FROM system.access.table_lineage WHERE target_table_catalog='{CATALOG}'
          AND target_table_schema='{SCHEMA}' ORDER BY event_time DESC LIMIT 50""")
        linaje = [fila.asDict() for fila in consulta_linaje.collect()]
        display(consulta_linaje)
    except Exception as e:
        acceso_linaje = type(e).__name__
        print("Linaje del sistema no consultable con esta identidad:",type(e).__name__,
              "· Catalog Explorer → tabla → Lineage.")
    from databricks.sdk import WorkspaceClient
    w=WorkspaceClient()
    j=job_equipo(w)
    print("Programación vigente",j.settings.schedule.as_dict() if j.settings.schedule else "sin schedule")
    import json
    from itertools import islice
    from pathlib import Path
    etapas = {"01_ingesta", "02_bronze", "03_silver", "04_gold", "05_oceanwatch"}
    por_clave = {t.task_key:t.notebook_task.notebook_path for t in j.settings.tasks if t.notebook_task}
    registros = []
    for r in islice(w.jobs.list_runs(job_id=j.job_id, expand_tasks=True, limit=12),12):
        tareas = []
        for rt in r.tasks or []:
            ruta = rt.notebook_task.notebook_path if rt.notebook_task else por_clave.get(rt.task_key, "")
            tareas.append({"clave":rt.task_key,"etapa":ruta.rsplit("/",1)[-1].removesuffix(".ipynb"),
                           "estado":rt.state.result_state.value if rt.state and rt.state.result_state else None})
        exitosas = {x["etapa"] for x in tareas if x["estado"]=="SUCCESS"}
        registros.append({"run_id":r.run_id, "trigger":r.trigger.value if r.trigger else None,
                          "estado":r.state.result_state.value if r.state and r.state.result_state else None,
                          "url":r.run_page_url,"tareas":tareas,"cinco_etapas_exitosas":etapas.issubset(exitosas)})
    evidencia = {"fecha_ais":fecha,"spark":spark.version,"workspace":w.config.host,"job_id":j.job_id,
                 "schedule":j.settings.schedule.as_dict() if j.settings.schedule else None,
                 "posiciones_plata":plata,"posiciones_trafico":traf,"posiciones_trayectorias":tray,
                 "linaje_observado":linaje,"consulta_linaje":acceso_linaje,
                 "ejecuciones":registros,"periodica_exitosa_observada":any(
                     r["trigger"]=="PERIODIC" and r["estado"]=="SUCCESS" and r["cinco_etapas_exitosas"] for r in registros)}
    destino = Path(VOLUME_ROOT)/"evidencias"/fecha/"verificacion.json"
    destino.parent.mkdir(parents=True,exist_ok=True)
    destino.write_text(json.dumps(evidencia,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    print("Evidencia de esta verificación:",destino)


In [ ]:
if modo == "configurar_job":
    configurar_job(dbutils.widgets.get("ruta_src").strip())
elif modo == "verificar":
    verificar()

## Bono · Proveedor de Open Sharing
Este modo publica las cuatro tablas Delta del contrato bajo el esquema compartido `oro`. Un destinatario de token se activa mediante un archivo de credenciales de un solo receptor. La firma de control del día se puede comparar en el notebook Free Edition. Ejecutar con permisos sobre el share y sus tablas.

In [ ]:
SHARE = "oceanwatch_eq8_oro"
RECIPIENT = "oceanwatch_eq8_free"

def publicar_share():
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.sharing import AuthenticationType
    w = WorkspaceClient()
    spark.sql(f"CREATE SHARE IF NOT EXISTS {SHARE} COMMENT 'Productos diarios OceanWatch del equipo 8'")
    # Sin sharing identifier se crea un destinatario OPEN de token y archivo .share.
    spark.sql(f"CREATE RECIPIENT IF NOT EXISTS {RECIPIENT} COMMENT 'Receptor independiente Free Edition'")
    receptor = w.recipients.get(RECIPIENT)
    if receptor.authentication_type != AuthenticationType.TOKEN:
        raise ValueError("El destinatario existe con otro mecanismo. Se requiere Open Sharing con token.")
    inventario = {r['name']: r['shared_object'] for r in spark.sql(f"SHOW ALL IN SHARE {SHARE}").collect()}
    esperados = {}
    for tabla in OROS:
        fisico = tabla.replace("`","")
        alias = "oro." + fisico.rsplit(".", 1)[-1]
        esperados[alias] = fisico
    if any(k not in esperados or v != esperados[k] for k,v in inventario.items()):
        raise ValueError("El share existente contiene un objeto fuera del contrato del equipo.")
    for alias, fisico in esperados.items():
        if alias not in inventario:
            spark.sql(f"ALTER SHARE {SHARE} ADD TABLE {fisico} AS {alias}")
    spark.sql(f"GRANT SELECT ON SHARE {SHARE} TO RECIPIENT {RECIPIENT}")
    display(spark.sql(f"SHOW ALL IN SHARE {SHARE}"))
    display(spark.sql(f"SHOW GRANTS ON SHARE {SHARE}"))
    print("Catalog Explorer → OpenSharing / Delta Sharing → Shared by me → Recipients →",
          RECIPIENT, "→ Activation link. El receptor descarga el perfil de uso privado.")

In [ ]:
def firma_diaria(tablas, dia):
    import datetime
    from databricks.sdk import WorkspaceClient
    iso = datetime.date.fromisoformat(str(dia)).isoformat()
    resultado = {}
    for tabla in tablas:
        # Misma receta en ambos workspaces. Las cuatro huellas solo contienen agregados.
        cols = spark.table(tabla).columns
        campos = ",".join("`"+c.replace("`","``")+"`" for c in cols)
        f = spark.sql(f"""SELECT count(*) n, coalesce(sum(cast(xxhash64({campos}) as decimal(38,0))),0) h
                        FROM {tabla} WHERE fecha=DATE '{iso}'""").first()
        resultado[tabla.replace("`","").split(".")[-1]] = {"filas":int(f.n),"suma_hash":str(f.h)}
    return {"workspace":WorkspaceClient().config.host.rstrip("/"), "fecha":iso, "tablas":resultado}

In [ ]:
if modo == "compartir":
    import json
    publicar_share()
    print(json.dumps(firma_diaria(OROS, fecha),ensure_ascii=False))